In [7]:
# Install complete dependencies if running in a clean runtime environment
!pip install --quiet pymupdf opencv-python-headless "numpy<2.0.0" paddlepaddle==2.6.2 paddleocr==2.8.1 ollama pydantic pillow matplotlib

In [8]:
# 1. Install the Ollama CLI wrapper if running in a cloud notebook (e.g., Google Colab)
!sudo apt-get install zstd

!curl -fsSL https://ollama.com/install.sh | sh

# 2. Start the Ollama background daemon service
import subprocess
import time

print("Starting Ollama background server...")
# Launch the server without blocking the notebook execution
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Give the server a few seconds to fully initialize
time.sleep(5)
print("Ollama server should be live.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (2,866 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 118243 files and directories currently 

In [9]:
# Pull the specific multimodal model required by your script
!ollama pull qwen2.5vl:7b

In [10]:
# Check local model availability
!ollama list

NAME            ID              SIZE      MODIFIED               
qwen2.5vl:7b    5ced39dfa4ba    6.0 GB    Less than a second ago    


In [11]:
import os
import sys
import cv2
import time
import json
import base64
import logging
import numpy as np
import fitz  # PyMuPDF
from PIL import Image
from paddleocr import PaddleOCR
import ollama
from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional, Dict, Any, Tuple

# Configure production-style logs to trace pipeline performance
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("ClaimsPipeline")

# Mute deep learning framework noise
logging.getLogger("ppocr").setLevel(logging.ERROR)

In [12]:
class UnifiedClaimSchema(BaseModel):
    """The unified data structure expected by our FastAPI enterprise database routing layer."""
    document_type: str = Field(..., description="Classification mapping: 'Hospital Bill', 'Prescription', 'Lab Report', or 'Discharge Summary'")
    institution_name: str = Field(..., description="The name of the issuing hospital, clinic, pharmacy, or laboratory facility.")
    patient_name: str = Field(..., description="Full legal name of the patient.")
    service_date: str = Field(..., description="The document's primary date of service formatted as YYYY-MM-DD or raw fallback string.")
    clinical_summary: str = Field(..., description="Aggregated text tracking primary diagnoses, findings, reasons for visit, or symptoms.")
    line_items_or_treatments: List[str] = Field(..., description="Array of itemized procedures, medications, or lab test metrics found.")
    monetary_total: Optional[float] = Field(None, description="Total balance due or payment total. Null if not applicable to document type.")
    confidence_metrics_audit: Dict[str, float] = Field(..., description="Internal pipeline execution metadata tracing engine processing stages.")

In [16]:
import ollama

response = ollama.chat(
    model="qwen2.5vl:7b",
    messages=[
        {"role": "user", "content": "Hello"}
    ]
)

print(response["message"]["content"])

Hello! How can I assist you today? Is there something specific you would like to know or discuss?


In [17]:
class EnterpriseClaimsPipeline:
    """
    An end-to-end processing pipeline that transforms raw document uploads into
    clean, structured, and schema-validated JSON data for production claims systems.
    """
    def __init__(self, use_gpu: bool = False):
        logger.info("Initializing industrial PaddleOCR engine instance...")

        # Map your use_gpu boolean to the new device string parameter
        target_device = "gpu" if use_gpu else "cpu"

        # Updated to remove use_gpu and use modern parameters
        self.ocr_engine = PaddleOCR(
            use_textline_orientation=True,  # Replaces deprecated use_angle_cls
            lang='en',
            device=target_device,           # Replaces use_gpu
        )
        self.vlm_model_name = "qwen2.5vl:7b"
        logger.info(f"Pipeline linked to local VLM service container target: {self.vlm_model_name}")

    def _convert_pdf_to_images(self, pdf_path: str, dpi: int = 150) -> List[np.ndarray]:
        """PyMuPDF Tier: Validates the file and converts every page to a pixel array."""
        doc = fitz.open(pdf_path)
        images = []
        zoom = dpi / 72
        matrix = fitz.Matrix(zoom, zoom)

        for page in doc:
            pix = page.get_pixmap(matrix=matrix, alpha=False)
            img_data = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)
            # Match OpenCV BGR color layouts
            img_data = cv2.cvtColor(img_data, cv2.COLOR_RGB2BGR)
            images.append(img_data)

        doc.close()
        return images

    def _preprocess_image(self, img: np.ndarray) -> np.ndarray:
        """OpenCV Tier: Optimizes document images by improving contrast and aligning text lines."""
        # Step A: Convert to single-channel grayscale
        if len(img.shape) == 3:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        else:
            gray = img

        # Step B: Apply Contrast Limited Adaptive Histogram Equalization (CLAHE)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        contrast_enhanced = clahe.apply(gray)

        # Step C: Smooth out background speckle noise while keeping edges crisp
        denoised = cv2.medianBlur(contrast_enhanced, 3)

        # Step D: Deskew and level horizontal alignment fields
        _, binary = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        inv_binary = cv2.bitwise_not(binary)
        coords = np.column_stack(np.where(inv_binary > 0))
        if coords.size > 0:
            angle = cv2.minAreaRect(coords)[-1]
            angle = -(90 + angle) if angle < -45 else -angle
            h, w = img.shape[:2]
            center = (w // 2, h // 2)
            rot_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
            img = cv2.warpAffine(img, rot_matrix, (w, h), flags=cv2.INTER_CUBIC, borderValue=(255, 255, 255))

        return img

    def _run_ocr(self, img: np.ndarray) -> Tuple[str, float]:
        """PaddleOCR Tier: Extracts text tokens and averages line confidence layers."""
        raw_results = self.ocr_engine.ocr(img, cls=True)
        text_lines = []
        confidence_scores = []

        if raw_results and raw_results[0]:
            for block in raw_results[0]:
                text = block[1][0]
                conf = block[1][1]
                text_lines.append(text)
                confidence_scores.append(conf)

        full_text_block = "\n".join(text_lines)
        mean_confidence = float(np.mean(confidence_scores)) if confidence_scores else 1.0
        return full_text_block, mean_confidence

    def _run_vlm_reasoning(self, img: np.ndarray, grounded_text: str) -> str:
        """Ollama/Qwen2.5-VL Tier: Runs semantic data extraction on the image and text."""
        _, buffer = cv2.imencode('.png', img)
        b64_image = base64.b64encode(buffer).decode('utf-8')

        system_prompt = (
            "You are an enterprise medical insurance ingestion agent.\n"
            "Analyze the document layout image and the provided OCR text block to extract structured metrics.\n"
            "You must return ONLY a raw JSON payload fitting this exact structure:\n"
            "{\n"
            '  "document_type": "Hospital Bill | Prescription | Lab Report | Discharge Summary",\n'
            '  "institution_name": "string",\n'
            '  "patient_name": "string",\n'
            '  "service_date": "YYYY-MM-DD",\n'
            '  "clinical_summary": "string describing diagnosis or symptoms summary",\n'
            '  "line_items_or_treatments": ["string items"],\n'
            '  "monetary_total": 0.0 or null\n'
            "}\n"
            "CRITICAL: Do not include markdown code block syntax wrappers like ```json. Do not include introductory conversational commentary."
        )

        user_prompt = f"Extract the target insurance details. Ground your extraction using this verified OCR text reference block:\n{grounded_text}"

        response = ollama.chat(
            model=self.vlm_model_name,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt, 'images': [b64_image]}
            ],
            options={'temperature': 0.0}
        )
        return response['message']['content'].strip()

    def execute_pipeline(self, file_path: str) -> List[UnifiedClaimSchema]:
        """
        Orchestrates the entire document ingestion lifecycle.
        Measures processing latency and returns validated data objects.
        """
        pipeline_start_time = time.perf_counter()
        logger.info(f"🚀 Initializing processing lifecycle for document target: {file_path}")

        # Step 1: Document Ingestion & Rasterization Check
        ext = os.path.splitext(file_path)[1].lower()
        if ext == '.pdf':
            pages_matrices = self._convert_pdf_to_images(file_path)
        else:
            img = cv2.imread(file_path)
            pages_matrices = [img] if img is not None else []

        if not pages_matrices:
            raise ValueError(f"Pipeline aborted: Target input file unreadable or empty at context: {file_path}")

        validated_outputs_collection = []

        # Step 2: Loop and Process Pages
        for idx, raw_matrix in enumerate(pages_matrices):
            page_start = time.perf_counter()
            logger.info(f"Processing Page Segment Index: {idx + 1}/{len(pages_matrices)}")

            # Sub-Tier A: OpenCV Optimization
            clean_matrix = self._preprocess_image(raw_matrix)

            # Sub-Tier B: OCR Execution
            ocr_text, ocr_confidence = self._run_ocr(clean_matrix)

            # Sub-Tier C: VLM Inference
            raw_vlm_output = self._run_vlm_reasoning(clean_matrix, ocr_text)

            # Sub-Tier D: Pydantic Validation & Schema Enforcement
            try:
                # Strip markdown syntax wrappers if the model includes them despite instructions
                if raw_vlm_output.startswith("```json"):
                    raw_vlm_output = raw_vlm_output[7:]
                if raw_vlm_output.endswith("```"):
                    raw_vlm_output = raw_vlm_output[:-3]
                raw_vlm_output = raw_vlm_output.strip()

                parsed_json = json.loads(raw_vlm_output)

                # Append audit metadata to track pipeline health
                page_latency = time.perf_counter() - page_start
                parsed_json["confidence_metrics_audit"] = {
                    "paddle_ocr_mean_confidence": ocr_confidence,
                    "pipeline_processing_latency_seconds": round(page_latency, 3)
                }

                # Enforce structure using Pydantic schema validation
                validated_claim = UnifiedClaimSchema(**parsed_json)
                validated_outputs_collection.append(validated_claim)
                logger.info(f"✅ Page {idx + 1} fully validated. Type extracted: {validated_claim.document_type}")

            except (json.JSONDecodeError, ValidationError) as schema_fault:
                logger.error(f"❌ Structural Validation Crash at page segment context index {idx + 1}: {str(schema_fault)}")
                logger.debug(f"Faulty raw data context: {raw_vlm_output}")

        total_latency = time.perf_counter() - pipeline_start_time
        logger.info(f"🏁 Pipeline processing cycle complete. Total runtime: {total_latency:.3f} seconds.")
        return validated_outputs_collection

In [18]:
from PIL import Image, ImageDraw
import fitz  # PyMuPDF

def generate_synthetic_corpus():
    """Generates a comprehensive test set of multi-classification document files."""
    # 1. Document Target: Prescription
    rx_img = Image.new('RGB', (600, 250), color=(255, 255, 255))
    draw = ImageDraw.Draw(rx_img)
    draw.text((30, 20), "OAKRIDGE PHARMACY & HEALTHCARE", fill=(0,0,0))
    draw.text((30, 60), "Patient: John Harrison", fill=(0,0,0))
    draw.text((30, 90), "Rx: Atorvastatin 20mg Tablets (Take 1 daily at bedtime)", fill=(0,0,0))
    draw.text((30, 120), "Dr. Amanda Ross, MD - Date: 2026-01-15", fill=(0,0,0))
    rx_img.save("prescription_claim.png")

    # 2. Document Target: Lab Report
    lab_img = Image.new('RGB', (600, 250), color=(255, 255, 255))
    draw = ImageDraw.Draw(lab_img)
    draw.text((30, 20), "SILVER LAKE DIAGNOSTIC LABORATORIES", fill=(0,0,0))
    draw.text((30, 60), "Patient Name: John Harrison", fill=(0,0,0))
    draw.text((30, 90), "Panel: Hemoglobin A1C Diagnostic Panel", fill=(0,0,0))
    draw.text((30, 120), "Result Metric Index: 5.8% (Pre-diabetic Range Variant)", fill=(0,0,0))
    draw.text((30, 150), "Authorized Release Date: 2026-02-11", fill=(0,0,0))
    lab_img.save("lab_report.png")

    # 3. Document Target: Hospital Bill (PDF)
    pdf1 = fitz.open()
    p1 = pdf1.new_page(width=600, height=300)
    p1.insert_text((40, 40), "ST. JUDE CLINICAL CENTER INVOICE", fontsize=16)
    p1.insert_text((40, 90), "Patient Target Account: John Harrison")
    p1.insert_text((40, 120), "Inpatient Facility Day Stay Charges:  $1,500.00")
    p1.insert_text((40, 150), "Anesthesia Evaluation Distribution:    $750.00")
    p1.insert_text((40, 190), "AGGREGATED BALANCE OWED PAYABLE:      $2,250.00")
    p1.insert_text((40, 230), "Billing Closeout Date Field: 2026-03-01")
    pdf1.save("hospital_bill.pdf")
    pdf1.close()

    # 4. Document Target: Discharge Summary (PDF)
    pdf2 = fitz.open()
    p2 = pdf2.new_page(width=600, height=300)
    p2.insert_text((40, 40), "VALLEY VIEW GENERAL HOSPITAL DISCHARGE SUMMARY", fontsize=14)
    p2.insert_text((40, 80), "Patient: John Harrison")
    p2.insert_text((40, 110), "Admit Diagnosis: Acute Appendicitis Surgery Status")
    p2.insert_text((40, 140), "Intervention: Laparoscopic Appendectomy Execution successful")
    p2.insert_text((40, 170), "Discharge Disposition Condition: Stable, outpatient recovery")
    p2.insert_text((40, 210), "Date of Final Release Summary: 2026-03-05")
    pdf2.save("discharge_summary.pdf")
    pdf2.close()

    print("📁 Complete corporate claims simulation corpus successfully written to local disk runtime environments.")

# Execute corpus generation
generate_synthetic_corpus()

📁 Complete corporate claims simulation corpus successfully written to local disk runtime environments.


In [ ]:
# Initialize Pipeline Module
pipeline = EnterpriseClaimsPipeline(use_gpu=False)

# Build unified run queue targets
target_claims_queue = [
    "prescription_claim.png",
    "lab_report.png",
    "hospital_bill.pdf",
    "discharge_summary.pdf"
]

all_extracted_claims_records = []

print("Starting end-to-end processing run across our test batch...")
print("==================================================================================")

for claim_file in target_claims_queue:
    try:
        results = pipeline.execute_pipeline(claim_file)
        all_extracted_claims_records.extend(results)
    except Exception as pipeline_error:
        print(f"❌ Fatal System Exception processing file target '{claim_file}': {str(pipeline_error)}")

print("==================================================================================")
print("Pipeline complete. Displaying validated extractions:")

Starting end-to-end processing run across our test batch...
